# 05-0e — Une seule règle pour mesurer la longueur de description

**Navigation** : [Index du rayon](README.md) | [05-0 — générateurs symboliques](05-0-Generateurs-Symboliques.ipynb) | [05-0b — demoscene](05-0b-Demoscene-Intros-Procedurales-Python.ipynb) | [05-0c — low-complexity art](05-0c-Low-Complexity-Art-Python.ipynb) | [05-0d — GANs et NFT](05-0d-GANs-Portraits-Moment-NFT.ipynb)

Ce rayon mesure depuis quatre carnets la même intuition, héritée de la scène
demoscene et de Schmidhuber : **un artefact procédural coûte bien moins à
décrire qu'à stocker**. Les chiffres publiés sont spectaculaires — ×1030, ×2365,
×156 — et ils sont exacts. Le problème est ailleurs : **ils ne sont pas
comparables entre eux.** Trois carnets, trois règles de mesure, trois unités.

Ce carnet ne produit pas une quatrième mesure. Il prend les artefacts déjà
publiés par le rayon, leur applique **une seule règle**, et dit pour chacun des
trois chiffres d'origine s'il est reproduit, corrigé, ou seulement rapporté.

| Ce que ce carnet fait | Ce qu'il ne fait pas |
|---|---|
| ramener les trois conventions à des **bits**, des deux côtés du ratio | mesurer de nouveaux artefacts |
| confronter le modèle de coût de 05-0c à un **codec réel** | trancher laquelle des trois mesures est « la bonne » |
| fournir un **témoin négatif** que l'instrument doit mal classer | remplacer les carnets du rayon |

Part of [#16774](https://github.com/jsboige/CoursIA/issues/16774) ·
Join [#16741](https://github.com/jsboige/CoursIA/issues/16741) (fil MDL) ·
Grain [#23095](https://github.com/jsboige/CoursIA/issues/23095).

## 1. Les trois règles du rayon, telles qu'elles s'impriment

Relisons les sorties réelles des trois carnets — ce sont des chiffres cités, pas
recalculés ici.

| Carnet | Ce qu'il affirme | Comment il l'obtient |
|---|---|---|
| 05-0 | `Koch : description 12 octets-equivalents -> rendu 12 357 octets PNG`, ratio **×1030** | `rendu_koch / description_koch` : des **octets divisés par des caractères** |
| 05-0 | `Regle110 : description 9 bits -> rendu 2 661 octets PNG`, ratio **×2365** | `rendu_110 * 8 / description_110` : des **bits par des bits** |
| 05-0b | `plasma : source 205 o, source.xz 200 o, frames.xz 1 092 368 o`, ratio **×5462** | `frames.xz / prog.xz` : du **compressé par du compressé** |
| 05-0c | `rosace, six petales ecrits : 65 bits` ; `un petale + repete x6 : 19 bits` ; ratio **×156** | un **modèle de coût maison** (`cout`, `gamma_bits`) face à un LZMA réel |

Deux lignes du même carnet (05-0), imprimées l'une sous l'autre, ne sont donc
**pas dans la même unité** : ×1030 compare des octets à des caractères, ×2365
compare des bits à des bits. Le second est juste ; le premier surestime d'un
facteur ~8, le prix d'un caractère ASCII en bits.

Et un quatrième chiffre, que 05-0b publie lui-même, achève de disqualifier le
ratio comme propriété de l'artefact :

| Frames rendues | 8 | 16 | 32 | 64 | 128 |
|---|---|---|---|---|---|
| `xz/programme` (plasma) | ×810 | ×1 506 | ×2 862 | ×5 462 | ×10 435 |

Le même plasma, décrit par le même programme, passe de ×810 à ×10 435 selon
**combien de frames on rend**. Un ratio n'est donc pas une propriété de
l'artefact : c'est une propriété du **couple** (artefact, taille de rendu).
C'est pour cela qu'il faut un instrument, et pas un chiffre à retenir.

## 2. L'instrument : un code en deux parties

La théorie qui donne un sens unique au mot « longueur de description » est le
**principe MDL** (Rissanen, 1978), et il est déjà enseigné dans ce dépôt — voir
[`Compression/Compression-06-MDL-Kolmogorov-Python.ipynb`](../../../Compression/Compression-06-MDL-Kolmogorov-Python.ipynb) :
décrire des données, c'est payer **deux choses**, le modèle et les données sous
ce modèle.

$$L_{\text{total}}(D) = \underbrace{L(H)}_{\text{transmettre le modèle}} + \underbrace{L(D\mid H)}_{\text{transmettre les données sous } H}$$

Un code de Huffman rend cette décomposition **littérale** au lieu de
métaphorique : `L(H)` est le coût de transmettre les longueurs de code (une par
symbole), `L(D|H)` est le flux encodé sous ce code. C'est exactement le genre de
quantité qu'un « nombre de caractères » ou un « nombre de bits estimés » ne
donne pas.

Reste une exigence, et c'est celle que ce carnet refuse de violer : **le codec
n'est pas réécrit ici.** Il vient de la série `Compression`, via son pont
`symbol_codec_bridge.py`, qui extrait par analyse AST les fonctions du carnet
`Compression-02` — aucune longueur, aucune table n'y est recopiée. Et la
convention de coût du modèle, `cout_modele`, vient du carnet MDL lui-même.

In [1]:
# Chargement des organes : le codec de Compression-02, la convention de Compression-06.
# Aucune de ces deux fonctions n'est réécrite dans ce carnet.
import ast
import importlib.util
import json
import lzma
from pathlib import Path


def racine_du_depot(depart: Path | None = None) -> Path:
    # Remonte jusqu'au repertoire qui contient MyIA.AI.Notebooks.
    ici = (depart or Path.cwd()).resolve()
    for parent in [ici, *ici.parents]:
        if (parent / "MyIA.AI.Notebooks").is_dir():
            return parent
    raise RuntimeError("racine du depot introuvable depuis " + str(ici))


RACINE = racine_du_depot()
COMPRESSION = RACINE / "MyIA.AI.Notebooks" / "Compression"
RAYON = RACINE / "MyIA.AI.Notebooks" / "GenAI" / "Image" / "05-History"


def charger_module(nom: str, chemin: Path):
    spec = importlib.util.spec_from_file_location(nom, chemin)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


pont = charger_module("symbol_codec_bridge", COMPRESSION / "symbol_codec_bridge.py")
CARNET_MDL = COMPRESSION / "Compression-06-MDL-Kolmogorov-Python.ipynb"
cout_modele = pont.load_functions({"cout_modele"}, CARNET_MDL)["cout_modele"]

print("codec     :", pont.__file__.replace(str(RACINE) + "\\", ""))
print("convention:", CARNET_MDL.name, "->", f"cout_modele(k=4, b=4) = {cout_modele(4, 4)} bits")

codec     : MyIA.AI.Notebooks\Compression\symbol_codec_bridge.py
convention: Compression-06-MDL-Kolmogorov-Python.ipynb -> cout_modele(k=4, b=4) = 20 bits


### Lecture.

`cout_modele(4, 4)` rend 20 bits : 8 bits d'en-tête, puis `k-1 = 3` longueurs à
`b = 4` bits. C'est bien la formule du carnet MDL qui s'exécute, importée par
analyse AST — le pont ne recopie ni la formule ni la table. Le témoin de ce
chargement est qu'une fonction disparue du carnet ferait lever le pont au lieu
de rendre un substitut : l'instrument ne peut pas mentir par défaut.

Notez aussi ce qui n'a **pas** été importé : `huffman`, `encoder`, `decoder`
sont exposés par le pont et utilisés tels quels, sans réimplémentation.

## 3. Les deux autres pièces : lire le programme, mesurer un flux

Pour appliquer une règle unique, il faut deux gestes que le rayon pratique déjà
mais chacun à sa façon :

1. **Lire le programme** qui produit l'artefact — non pas une version retapée,
   mais le texte source réellement embarqué dans le carnet, extrait par AST.
2. **Mesurer un flux** en bits, avec le même codec des deux côtés.

Le geste 1 reprend le principe du pont : on sélectionne **par nom**, jamais par
index — par nom de fonction pour le code, et par les **constantes qu'elle définit**
pour la cellule d'installation (`W`, `H`, `N_FRAMES`). Une renumérotation du
carnet ne casse alors ni l'un ni l'autre : rien ne dépend d'un rang.

In [2]:
def cellules_code(carnet: Path) -> list[str]:
    data = json.loads(Path(carnet).read_text(encoding="utf-8"))
    return ["".join(c["source"]) for c in data["cells"] if c["cell_type"] == "code"]


def source_de(carnet: Path, *noms: str) -> str:
    # Le texte source exact des fonctions nommees, tel qu'il est dans le carnet.
    trouve: dict[str, str] = {}
    for source in cellules_code(carnet):
        try:
            arbre = ast.parse(source)
        except SyntaxError:
            continue
        for noeud in arbre.body:
            if isinstance(noeud, ast.FunctionDef):
                trouve[noeud.name] = ast.get_source_segment(source, noeud)
    manquants = [n for n in noms if n not in trouve]
    if manquants:
        raise KeyError(f"fonctions absentes de {Path(carnet).name} : {manquants}")
    return "\n\n".join(trouve[n] for n in noms)


def xz_bits(donnees: bytes) -> int:
    # Le codec de reference de la serie Compression : LZMA au preset extreme.
    return 8 * len(lzma.compress(donnees, preset=9 | lzma.PRESET_EXTREME))


def deux_parties(donnees: bytes) -> dict:
    # Code en deux parties : L(H) + L(D|H), sur le codec reel de Compression-02.
    poids: dict[int, int] = {}
    for octet in donnees:
        poids[octet] = poids.get(octet, 0) + 1
    longueurs = pont.canonical_lengths(poids)
    code = pont.canonical_code(longueurs)
    if not (pont.is_prefix_free(code) and pont.is_injective(code)):
        raise AssertionError("le codec du pont n'est plus un code prefixe injectif")
    flux = pont.encoder(donnees, code)
    k = len(longueurs)
    b = max(1, max(longueurs.values()).bit_length())
    l_h, l_d = cout_modele(k, b), len(flux)
    return {"k": k, "b": b, "L(H)": l_h, "L(D|H)": l_d,
            "total": l_h + l_d, "xz": xz_bits(donnees), "brut": 8 * len(donnees)}


exemple = deux_parties(b"plasma plasma plasma")
print(f"temoin : {exemple['brut']} bits bruts, deux parties {exemple['total']} bits "
      f"(modele {exemple['L(H)']} + flux {exemple['L(D|H)']}), xz {exemple['xz']} bits")

temoin : 160 bits bruts, deux parties 69 bits (modele 18 + flux 51), xz 608 bits


### Lecture.

Sur une chaîne très répétitive, le total en deux parties est **plus petit** que
le brut : c'est le codec qui travaille. Le modèle (`L(H)`) coûte ici une part
visible du total, et c'est précisément le terme que les trois règles du rayon
comptent chacune à leur manière — ou oublient. Un « nombre de caractères » ne
paie aucun modèle ; un `lzma(source)` paie un modèle implicite (sa table de
contexte) sans le dire ; le modèle maison de 05-0c paie un modèle explicite mais
dont aucun codec ne fixe le prix.

## 4. La même règle appliquée aux sept programmes du rayon

Chaque artefact du rayon est produit par une ou plusieurs fonctions définies
dans un carnet. Ce sont ces fonctions, et elles seules, qui constituent la
**description** de l'artefact : le rendu n'a aucun contenu propre, il est
recalculé. On les mesure donc en bits, deux fois : par le code en deux parties
(le modèle est payé explicitement) et par LZMA (le modèle est implicite).

In [3]:
PROGRAMMES = [
    ("05-0b", "plasma", RAYON / "05-0b-Demoscene-Intros-Procedurales-Python.ipynb", ("plasma",)),
    ("05-0b", "tunnel", RAYON / "05-0b-Demoscene-Intros-Procedurales-Python.ipynb", ("tunnel",)),
    ("05-0b", "feu", RAYON / "05-0b-Demoscene-Intros-Procedurales-Python.ipynb", ("feu",)),
    ("05-0b", "rotozoom", RAYON / "05-0b-Demoscene-Intros-Procedurales-Python.ipynb", ("rotozoom",)),
    ("05-0b", "raymarching", RAYON / "05-0b-Demoscene-Intros-Procedurales-Python.ipynb", ("scene_sdf", "raymarch")),
    ("05-0", "regle 110", RAYON / "05-0-Generateurs-Symboliques.ipynb", ("regle110",)),
    ("05-0c", "modele de cout", RAYON / "05-0c-Low-Complexity-Art-Python.ipynb", ("gamma_bits", "cout")),
]

mesures = []
print(f"{'carnet':8s}{'programme':16s}{'source':>8s}{'L(H)':>7s}{'L(D|H)':>8s}"
      f"{'total':>8s}{'xz':>8s}")
for carnet, nom, chemin, fonctions in PROGRAMMES:
    texte = source_de(chemin, *fonctions)
    r = deux_parties(texte.encode("utf-8"))
    mesures.append((carnet, nom, len(texte), r))
    print(f"{carnet:8s}{nom:16s}{len(texte):>8d}{r['L(H)']:>7d}{r['L(D|H)']:>8d}"
          f"{r['total']:>8d}{r['xz']:>8d}")

carnet  programme         source   L(H)  L(D|H)   total      xz
05-0b   plasma               204    156     853    1009    1600
05-0b   tunnel               504    196    2068    2264    2752
05-0b   feu                  532    200    2484    2684    3232


05-0b   rotozoom             348    176    1564    1740    2080
05-0b   raymarching         1224    220    5479    5699    5024


05-0    regle 110            419    188    1916    2104    2656
05-0c   modele de cout       217    156    1010    1166    1888


### Lecture.

Deux choses se lisent dans ce tableau, et aucune ne se voyait dans les ratios.

**Premièrement, le classement n'est pas celui des octets.** `raymarching` a la
plus longue source (1 224 caractères) mais son code en deux parties dépasse
son LZMA sur une ligne sur sept (5 699 contre 5 024 bits) : sur un texte long et
varié, le modèle d'ordre 0 du code de Huffman perd contre la mémoire de contexte
de LZMA. Cette rareté est le fait intéressant — le cas n'est pas la règle. Inversement, sur les sources
courtes et redondantes (`plasma`, 204 caractères ici, contre 205 octets dans 05-0b
: le même bloc de quatre lignes, mais `inspect.getsource` garde le saut de ligne final
et `ast.get_source_segment` non), les deux parties gagnent. Le
choix du codec n'est donc pas neutre : il faut le **dire**, pas le subir — et
c'est une raison de plus pour que les deux côtés d'un ratio soient mesurés par
**le même** codec.

**Deuxièmement, `L(H)` n'est jamais négligeable.** Entre 156 et 220 bits pour un
programme de 204 à 1 224 caractères : le modèle pèse de **3,9 %** (raymarching,
220 sur 5 699) à **15,5 %** (plasma, 156 sur 1 009) du total. Un
ratio qui compare un « nombre de caractères » à des octets de PNG ne paie jamais
ce terme.

## 5. Le côté rendu, et le témoin négatif

Comparer des longueurs de programme ne suffit pas : le rayon compare un
*programme* à un *rendu*. Il faut donc mesurer aussi un rendu, avec le même
codec, et — c'est le point que l'organ-first impose — fournir un **témoin
négatif** : un artefact de description courte et de contenu incompressible, que
l'instrument doit **mal classer**. S'il le classait bien, il mesurerait la
longueur du programme, pas la complexité du résultat.

Le plasma se régénère exactement comme le carnet 05-0b le fait : on exécute sa
cellule d'installation (constantes `W`, `H`, `N_FRAMES`, grilles `x`, `y`), puis
on charge la seule fonction `plasma` par AST. Aucune valeur n'est retapée.

In [4]:
import numpy as np

CARNET_05B = RAYON / "05-0b-Demoscene-Intros-Procedurales-Python.ipynb"


def noms_definis(source: str) -> set[str]:
    # Les noms qu'une cellule DEFINIT (affectation simple, multiple ou annotee).
    definis: set[str] = set()
    for noeud in ast.parse(source).body:
        if isinstance(noeud, ast.Assign):
            cibles = noeud.targets
        elif isinstance(noeud, ast.AnnAssign):
            cibles = [noeud.target]
        else:
            continue
        for cible in cibles:
            for sous in ast.walk(cible):
                if isinstance(sous, ast.Name):
                    definis.add(sous.id)
    return definis


def cellule_qui_definit(carnet: Path, requis: set[str]) -> str:
    # Selection par ce que la cellule DEFINIT, jamais par son rang : c'est la
    # meme regle que pour les fonctions, ou l'on selectionne par nom. Une
    # renumeroation du carnet ne peut donc pas casser ce chargement.
    for source in cellules_code(carnet):
        if requis <= noms_definis(source):
            return source
    raise LookupError(f"aucune cellule ne definit {sorted(requis)} dans {carnet.name}")


installation = cellule_qui_definit(CARNET_05B, {"W", "H", "N_FRAMES"})
espace: dict = {}
exec("".join(installation), espace)          # imports et constantes du carnet, seulement

# `plasma` depend de x, y, W, H : le pont ne charge que des fonctions, pas des
# variables. On charge donc la definition par AST et on la lie a l'espace du carnet.
for source in cellules_code(CARNET_05B):
    arbre = ast.parse(source)
    for noeud in arbre.body:
        if isinstance(noeud, ast.FunctionDef) and noeud.name == "plasma":
            exec(compile(ast.Module(body=[noeud], type_ignores=[]), "<plasma>", "exec"), espace)
plasma = espace["plasma"]

frames = np.stack([plasma(t) for t in range(espace["N_FRAMES"])])
rng = np.random.default_rng(7)
bruit = rng.integers(0, 256, size=frames.shape, dtype=np.uint8)

for nom, flux in (("plasma 64 frames", frames), ("bruit 64 frames", bruit)):
    bits = xz_bits(flux.tobytes())
    print(f"{nom:18s} brut {8 * flux.nbytes:>10,d} bits -> xz {bits:>10,d} bits "
          f"({bits / (8 * flux.nbytes):.3f} bit/bit brut)")

print()
print("reproduction du chiffre publie par 05-0b :",
      f"{xz_bits(frames.tobytes()) // 8:,d} octets xz pour plasma 64 frames "
      f"(le carnet publie 1,092,368)")

Resolution 320x200, 64 frames -> 4,096,000 octets bruts par sequence


plasma 64 frames   brut 32,768,000 bits -> xz  8,738,944 bits (0.267 bit/bit brut)


bruit 64 frames    brut 32,768,000 bits -> xz 32,770,144 bits (1.000 bit/bit brut)



reproduction du chiffre publie par 05-0b : 1,092,368 octets xz pour plasma 64 frames (le carnet publie 1,092,368)


### Lecture.

La reproduction est exacte : le carnet 05-0b publie `frames.xz 1 092 368` octets
pour le plasma à 64 frames, et le même codec, appliqué ici au rendu régénéré,
retrouve ce nombre. C'est le contrôle qui autorise à comparer : l'instrument
unique n'est pas un instrument concurrent, c'est **le même**, appliqué des deux
côtés.

Le témoin négatif ne dit pas ce qu'on attendait de lui, et c'est instructif. Le
plasma tombe à 0,27 bit par bit brut ; le bruit reste à 1,00 — sur le **rendu**,
l'instrument sépare donc nettement les deux. Mais le carnet 05-0c, sur les mêmes
objets, mesure 65 bits pour la rosace et 113 bits pour le dessin `hasard` — un
écart de 1,7× — et son ratio publié ne diffère que de ×156 à ×93. Autrement dit
les deux mesures s'accordent sur un point qui saute aux yeux une fois les
chiffres alignés : **ce n'est pas le rendu qui distingue la rosace du hasard**
(10 144 bits contre 10 560, soit 4 % d'écart — deux images 1 bit de même
entropie), **c'est la description**. Un ratio qui divise un rendu par une
description hérite donc presque entièrement de la mesure de description — et
c'est précisément la grandeur que les trois carnets mesurent différemment.

## 6. Le tableau unique — et le verdict sur chacun des trois chiffres

Tout est désormais dans une seule unité, le bit. Une seule règle gouverne le
tableau : **les deux côtés d'un ratio sont mesurés par le même codec**, et la
provenance de chaque nombre est déclarée — `MESURE` s'il est recalculé ici,
`RAPPORTE` s'il est cité de la sortie d'un carnet du rayon.

In [5]:
# Description de Koch telle que 05-0 la construit : axiome + regle + deux angles.
# 05-0 compte `len('F') + len('F+F-F-F+F') + 2`, soit 12 unites : les deux angles
# sont des constantes du programme, pas des caracteres de la chaine.
description_koch = "F" + "F+F-F-F+F"
description_koch_caracteres = len(description_koch) + 2
r_koch = deux_parties(description_koch.encode("utf-8"))
rendu_koch_octets = 12_357          # RAPPORTE : sortie du carnet 05-0
rendu_koch_bits = 8 * rendu_koch_octets

# Regle 110 : table de 8 sorties binaires + une cellule graine, quantite deja en bits.
description_110_bits = 8 + 1        # RAPPORTE : convention du carnet 05-0
rendu_110_bits = 8 * 2_661          # RAPPORTE : sortie du carnet 05-0

plasma_source = dict((nom, r) for _, nom, _, r in mesures)["plasma"]
rendu_plasma_bits = xz_bits(frames.tobytes())   # MESURE

lignes = [
    ("05-0 Koch (comme publie)", "octets / caracteres",
     description_koch_caracteres, f"x{rendu_koch_octets / description_koch_caracteres:.0f}",
     f"reproduit le x1030 publie ; xz du texte = {r_koch['xz']} b"),
    ("05-0 Koch (deux parties)", "bits / bits",
     r_koch["total"], f"x{rendu_koch_bits / r_koch['total']:.0f}",
     "description MESURE, rendu RAPPORTE"),
    ("05-0 regle 110", "bits / bits (deja juste)",
     description_110_bits, f"x{rendu_110_bits / description_110_bits:.0f}",
     "reproduction du chiffre publie x2365"),
    ("05-0b plasma 64 f", "xz / xz",
     plasma_source["xz"], f"x{rendu_plasma_bits / plasma_source['xz']:.0f}",
     "les deux cotes MESURE"),
]

print(f"{'artefact':26s}{'regle':24s}{'description':>12s}{'ratio':>10s}  provenance")
for nom, regle, desc, ratio, prov in lignes:
    print(f"{nom:26s}{regle:24s}{desc:>10d} b{ratio:>10s}  {prov}")

artefact                  regle                    description     ratio  provenance
05-0 Koch (comme publie)  octets / caracteres             12 b     x1030  reproduit le x1030 publie ; xz du texte = 544 b
05-0 Koch (deux parties)  bits / bits                     26 b     x3802  description MESURE, rendu RAPPORTE
05-0 regle 110            bits / bits (deja juste)         9 b     x2365  reproduction du chiffre publie x2365
05-0b plasma 64 f         xz / xz                       1600 b     x5462  les deux cotes MESURE


### Lecture.

Trois verdicts, et deux seulement sont des chiffres à retenir tels quels.

1. **×1030 (05-0, Koch)** : le chiffre se reproduit **exactement** — `12 357`
   octets divisés par les `12` unités de description que 05-0 compte (axiome,
   règle, deux angles) redonnent bien 1 030. Mais c'est un quotient d'**octets
   par des caractères**, et le défaut n'est pas seulement l'unité : **le rendu de
   Koch n'a jamais été mesuré sous le codec de sa description**. Il n'existe donc
   pas, dans ce que le rayon publie, de quoi former un ratio à règle unique pour
   cet artefact. La correction va d'ailleurs dans le sens inverse de l'intuition :
   sous le code en deux parties, une description qui tient en `F` et
   `F+F-F-F+F` coûte **26 bits**, pas les 96 d'une conversion caractère → bit —
   et le ratio rapporté passe de ×1 030 à ×3 802. Un artisanat de 12 unités
   n'était pas une mesure prudente : c'était une mesure **par en dessous**, sur
   un artefact dont la description est presque entièrement du code répétitif.
2. **×2365 (05-0, règle 110)** : juste. `8 + 1` bits est bien une description en
   bits — la table d'un automate élémentaire tient en 8 bits, la graine en 1 — et
   le rendu est converti en bits. La règle divise des bits par des bits.
3. **×5462 (05-0b, plasma)** : juste, et c'est le seul qui soit
   **reproductible à l'identique** : le même codec retrouve 1 092 368 octets sur
   le rendu régénéré. Ce ratio reste vrai — à condition de dire **pour combien de
   frames**, puisque le même plasma passe de ×810 à ×10 435 de 8 à 128 frames.

Ce qui n'apparaît nulle part, et que ce carnet nomme : **05-0c ne mesure pas la
même chose que les autres.** Ses 65 bits sont le prix d'un *programme* sous un
modèle maison, et son ×156 compare ce prix de programme à un LZMA de *rendu*. Ce
n'est pas une erreur, c'est une autre question — et aucune des trois lignes ne
peut donc servir de base à une comparaison entre artefacts du rayon.

Un détail d'instrument, enfin, qui mérite d'être dit parce qu'il piège : sur une
description de douze caractères, `lzma` est un **mauvais** codec — son en-tête
coûte plus que le texte. Le tableau imprime pour Koch le code en deux parties,
pas LZMA, et le compte-rendu du xz est donné à côté pour que l'écart se voie au
lieu de se deviner.

## 7. Exercices

**Exercice 1 — Le modèle de coût tient-il sa promesse ?**
Le carnet 05-0c affirme qu'un pétale répété six fois se décrit en 19 bits contre
65 pour les six pétales écrits. Prenez deux dessins du rayon, mesurez les deux
sous le code en deux parties, et dites si la réduction observée sur le
*programme* se retrouve sur le *rendu*. Une réduction de programme qui ne
change pas le rendu est une information, pas un échec — expliquez laquelle.

**Exercice 2 — Où se cache le modèle dans LZMA ?**
`lzma.compress` ne publie pas de `L(H)`. Trouvez une façon de majorer le coût de
son modèle (par exemple en comptant l'en-tête du flux, ou en comparant deux
presets) et dites si ce coût suffit à expliquer les écarts du tableau de la
section 4.

**Exercice 3 — Un artefact à mi-chemin.**
Construisez un artefact dont le programme est court mais le rendu partiellement
incompressible — par exemple un plasma auquel on ajoute un bruit de faible
amplitude. Mesurez le rapport entre l'énergie du bruit et le taux de
compression du rendu. L'instrument doit-il le classer plutôt comme le plasma ou
plutôt comme le témoin ?

In [6]:
# Exercice 1
resultat_exercice_1 = None  # TODO etudiant

# Exercice 2
resultat_exercice_2 = None  # TODO etudiant

# Exercice 3
partiellement_bruite = None  # TODO etudiant
resultat_exercice_3 = None  # TODO etudiant

print("Exercices a completer")

Exercices a completer


## Ce que ce carnet a établi

| Résultat | Mesure |
|---|---|
| Une règle unique existe et s'applique | code en deux parties `L(H) + L(D\|H)` sur le codec de Compression-02, convention `cout_modele` de Compression-06 |
| Le codec du rayon est **le même** que celui de la série Compression | plasma régénéré : `1 092 368` octets xz, chiffre publié par 05-0b retrouvé |
| Le témoin négatif est distingué | plasma 0,27 bit/bit brut, bruit 1,00 |
| ×2365 et ×5462 sont justes | bits/bits et xz/xz |
| ×1030 se reproduit mais n'est pas récupérable | octets ÷ caractères ; sous le code en deux parties la description pèse 26 bits, donc ×1 030 devient ×3 802 — et le rendu reste une valeur rapportée |
| Ce n'est pas le rendu qui sépare rosace et hasard | 10 144 contre 10 560 bits, 4 % ; la différence des ratios (×156 contre ×93) vient des descriptions (65 contre 113 bits) |
| Un ratio n'est pas une propriété de l'artefact | plasma : ×810 à 8 frames, ×10 435 à 128 |

**Limites assumées.** Le côté rendu n'est mesuré ici que pour le plasma ; les
rendus de 05-0 (turtle, PNG) et de 05-0c (bitmap d'arcs) sont **cités**, pas
recalculés, et le tableau le dit. Le code en deux parties est appliqué aux
descriptions, jamais aux rendus de plusieurs mégaoctets : l'encodeur du pont est
en Python pur, et un flux de quatre millions de symboles n'y a pas sa place dans
un carnet pédagogique — c'est une limite de l'outil, dite ici plutôt que
contournée par un chiffre inventé.

**Ce que ce carnet ne prétend pas.** Il ne dit pas laquelle des trois mesures du
rayon est « la bonne » : il dit ce que chacune price réellement, et à quelle
condition deux d'entre elles peuvent être mises côte à côte.